In [11]:
import tensorflow as tf
import tensorflow_hub as hub
import pandas as pd
import numpy as np
import csv
import os

import matplotlib.pyplot as plt
from IPython.display import Audio
from scipy.io import wavfile

# Загрузка модели YAMNET
model = hub.load('https://tfhub.dev/google/yamnet/1')

In [12]:
# Путь к папке с аудиофайлами
audiofiles_folder = "/Users/nikitavdovichev/Documents/5 курс/ПроектнаяПрактика/dataset/audiofiles_16khz_mono"

# Проверим что все файлы нужного формата
for root, dirs, files in os.walk(audiofiles_folder):
    for file in files:
        if file.endswith(".wav"):
            file_path = os.path.join(root, file)
            
            try:
                # Декодирование аудиофайла с помощью TensorFlow
                wav_data, sr = tf.audio.decode_wav(tf.io.read_file(file_path), desired_channels=1)
                # Проверка частоты дискретизации
                assert sr == 16000, f"Файл {file_path} имеет неверную частоту дискретизации: {sr}"
                # Проверка на моно формат
                assert wav_data.shape[1] == 1, f"Файл {file_path} не является монофайлом"
                # print(f"File {file_path} is valid.")
            
            except Exception as e:
                print(f"Ошибка обработки файла {file_path}: {e}")

print("Все файлы соответствуют требованиям!")

Все файлы соответствуют требованиям!


In [13]:
def ensure_sample_rate(original_sample_rate, waveform,
                       desired_sample_rate=16000):
  """
  Ресемплирование аудио до нужной частоты дискретизации (16 кГц для YAMNet).
  
  Если исходная частота не равна желаемой, сигнал передискретизируется.
  """
  if original_sample_rate != desired_sample_rate:
    # Вычисляем новую длину сигнала после ресемплирования
    desired_length = int(round(float(len(waveform)) /
                               original_sample_rate * desired_sample_rate))
    # Применяем ресемплирование
    waveform = scipy.signal.resample(waveform, desired_length)
  return desired_sample_rate, waveform


def class_names_from_csv(class_map_csv_text):
  """
  Загружает человекочитаемые названия классов из CSV-файла YAMNet.
  
  YAMNet классифицирует 521 тип звуков, каждый с уникальным названием.
  """
  class_names = []
  # Открываем файл с маппингом классов
  with tf.io.gfile.GFile(class_map_csv_text) as csvfile:
    reader = csv.DictReader(csvfile)
    for row in reader:
      # Добавляем название класса из поля 'display_name'
      class_names.append(row['display_name'])

  return class_names

# Получаем путь к файлу с описанием классов из модели
class_map_path = model.class_map_path().numpy()

# Загружаем названия всех классов
class_names = class_names_from_csv(class_map_path)

In [14]:
# Путь к  CSV файлу
metadata_csv = r"/Users/nikitavdovichev/Documents/5 курс/ПроектнаяПрактика/dataset/UrbanSound8K_edited.csv"
df = pd.read_csv(metadata_csv)

# Папка с аудиофайлами
audiofiles_folder = "/Users/nikitavdovichev/Documents/Work/grant-0-shoot/dataset/UrbanSound8K/audiofiles_16khz_mono"

# Список классов, связанных с выстрелами (какие может предсказать модель)
gunshot_classes = [
    'Gunshot, gunfire',
    'Machine gun',
    'Explosion',
    'Fusillade',
    'Artillery fire',
    'Cap gun',
    'Burst, pop',
    'Boom'
]

# Список для хранения результатов
yamnet_output = []

audiofiles_passed = 0
# Перебираем строки в DataFrame
for index, row in df.iterrows():
    wav_file_name = os.path.join(audiofiles_folder, row["slice_file_name"])
    
    # Проверяем, существует ли файл
    if not os.path.exists(wav_file_name):
        print(f"Не найден файл: {wav_file_name}")
        yamnet_output.append(0)  # Если файл отсутствует, добавляем 0
        continue
    
    # Читаем аудиофайл
    sample_rate, wav_data = wavfile.read(wav_file_name)
    
    # Проверяем частоту дискретизации
    sample_rate, wav_data = ensure_sample_rate(sample_rate, wav_data)
    
    # Нормализуем аудиоданные
    waveform = wav_data / tf.int16.max
    
    # Запускаем модель
    scores, embeddings, spectrogram = model(waveform)
    
    # Получаем предсказанный класс
    scores_np = scores.numpy()
    inferred_class = class_names[scores_np.mean(axis=0).argmax()]
    
    # Проверяем, относится ли класс к выстрелам
    if inferred_class in gunshot_classes:
        yamnet_output.append(1)
    else:
        yamnet_output.append(0)

    audiofiles_passed += 1
    if audiofiles_passed % 500 == 0:
        print(f"Обработано {audiofiles_passed} файлов")

# Добавляем столбец yamnet_is_gun_shot в DataFrame
df['yamnet_is_gun_shot'] = yamnet_output

# Сохраняем DataFrame в новый CSV файл
output_csv = r"/Users/nikitavdovichev/Documents/5 курс/ПроектнаяПрактика/dataset/UrbanSound8K_with_model_predicts.csv"
df.to_csv(output_csv, index=False)

print("Обработка завершена, результат сохранен в:", output_csv)

Обработано 500 файлов
Обработано 1000 файлов
Обработано 1500 файлов
Обработано 2000 файлов
Обработано 2500 файлов
Обработано 3000 файлов
Обработано 3500 файлов
Обработано 4000 файлов
Обработано 4500 файлов
Обработано 5000 файлов
Обработано 5500 файлов
Обработано 6000 файлов
Обработано 6500 файлов
Обработано 7000 файлов
Обработано 7500 файлов
Обработано 8000 файлов
Обработано 8500 файлов
Обработка завершена, результат сохранен в: /Users/nikitavdovichev/Documents/5 курс/ПроектнаяПрактика/dataset/UrbanSound8K_with_model_predicts.csv


In [15]:
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

# Загрузка данных из CSV
data = r"/Users/nikitavdovichev/Documents/5 курс/ПроектнаяПрактика/dataset/UrbanSound8K_with_model_predicts.csv"
df = pd.read_csv(data)
true_labels = df['is_gun_shot']  # Истинные метки
predicted_labels = df['yamnet_is_gun_shot']  # Предсказанные метки

# Precision, recall и F1
precision = precision_score(true_labels, predicted_labels)
recall = recall_score(true_labels, predicted_labels)
f1 = f1_score(true_labels, predicted_labels)

# Результаты
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

Precision: 0.9522
Recall: 0.5321
F1 Score: 0.6827
